# CPU training time

Issue: #30

## Question

Later phases may need training that runs unattended on CPU (no GPU; a container on a Mac cannot use the GPU either). How long does each neural model take to train on this Mac's CPU, how much memory does it need, and would a full-size run fit in about an hour?

## Method

Each model is trained in its **own fresh process** (so memory is measured per model and nothing else is loaded), on **CPU only** with 4 threads, at a fixed size and a fixed number of epochs:
- **Two-Tower:** 500,000 sampled training queries from the training window (days 0-13), 4 epochs, batch 2048, in-batch negatives with logQ. Also scored on the evaluation queries for recall@100.
- **DCN-V2 + MMoE:** the feature table of #22 (40,000 queries x 200 candidates, 90 percent used for training), 3 epochs, batch 4096.

Scripts: `notebooks/bench_two_tower.py` and `notebooks/bench_dcn.py`. Time is wall-clock seconds for training only, with data preparation reported separately. The hardware is read from the machine.

In [ ]:
import json
import platform
import subprocess
import sys
from pathlib import Path

import polars as pl

ROOT = Path.cwd() if (Path.cwd() / "notebooks").exists() else Path.cwd().parent
chip = subprocess.run(["sysctl", "-n", "machdep.cpu.brand_string"], capture_output=True, text=True).stdout.strip()
ram = int(subprocess.run(["sysctl", "-n", "hw.memsize"], capture_output=True, text=True).stdout.strip()) / 2**30
cores = subprocess.run(["sysctl", "-n", "hw.ncpu"], capture_output=True, text=True).stdout.strip()
hardware = {"chip": chip, "ram_gb": round(ram), "cores": int(cores), "os": platform.platform(), "torch_threads_used": 4}
print(hardware)

In [ ]:
def run(script):
    out = subprocess.run([sys.executable, str(ROOT / "notebooks" / script)], capture_output=True, text=True, cwd=ROOT)
    assert out.returncode == 0, out.stderr[-2000:]
    line = [ln for ln in out.stdout.splitlines() if ln.startswith("RESULT ")][-1]
    return json.loads(line[len("RESULT "):])


tt = run("bench_two_tower.py")
dcn = run("bench_dcn.py")
print(json.dumps(tt, indent=1))
print(json.dumps(dcn, indent=1))

## Measured table

In [ ]:
table = pl.DataFrame([
    {"model": tt["model"], "training size": f"{tt['queries']:,} queries ({tt['examples']:,} examples)", "epochs": tt["epochs"],
     "train seconds": tt["train_seconds"], "seconds/epoch": tt["seconds_per_epoch"], "setup seconds": tt["setup_seconds"],
     "peak memory GB": tt["peak_memory_gb"], "quality": f"validation loss {tt['best_validation_loss']}, eval recall@100 {tt['eval_recall100']}"},
    {"model": dcn["model"], "training size": f"{dcn['queries']:,} queries ({dcn['rows']:,} rows)", "epochs": dcn["epochs"],
     "train seconds": dcn["train_seconds"], "seconds/epoch": dcn["seconds_per_epoch"], "setup seconds": dcn["setup_seconds"],
     "peak memory GB": dcn["peak_memory_gb"], "quality": f"validation loss {dcn['best_validation_loss']}, AUC {dcn['validation_auc']}"},
])
with pl.Config(tbl_width_chars=220, fmt_str_lengths=80, tbl_cols=10):
    print(table)
assert tt["peak_memory_gb"] < 8 and dcn["peak_memory_gb"] < 8

## Estimate for a full-size run

Training time grows about linearly with the number of examples (Two-Tower) or rows (DCN), so the time per epoch measured above is scaled to the full training window. The whole window has 890,042 usable queries. Assumptions written down: Two-Tower examples per query as measured above; DCN uses 200 candidate rows per query as built in #22, and building those feature rows (11.3 million rows took 78 seconds in #22, so about 7 seconds per million) is added.

In [ ]:
FULL_QUERIES = 890_042
tt_examples_per_query = tt["examples"] / tt["queries"]
tt_full_examples = FULL_QUERIES * tt_examples_per_query
tt_per_example = tt["seconds_per_epoch"] / tt["examples"]
tt_epoch_full = tt_per_example * tt_full_examples
tt_full = 1 * tt_epoch_full          # the best epoch was epoch 1 in #24 and #25, so one to two epochs are enough
tt_full2 = 2 * tt_epoch_full

dcn_per_row = dcn["seconds_per_epoch"] / dcn["rows"]
dcn_rows_full = FULL_QUERIES * 200
dcn_epoch_full = dcn_per_row * dcn_rows_full
dcn_feature_build = dcn_rows_full / 1e6 * 7          # seconds, from #22
dcn_full_3 = 3 * dcn_epoch_full + dcn_feature_build
dcn_full_10 = 10 * dcn_epoch_full + dcn_feature_build
est = pl.DataFrame([
    {"model": "Two-Tower", "full-window training size": f"{FULL_QUERIES:,} queries, ~{tt_full_examples:,.0f} examples", "one epoch (min)": round(tt_epoch_full / 60, 1), "plan": "1 to 2 epochs (best epoch was 1)", "total (min)": f"{tt_full / 60:.0f} to {tt_full2 / 60:.0f}", "fits in ~60 min": "yes" if tt_full2 / 60 <= 60 else "no"},
    {"model": "DCN-V2 + MMoE", "full-window training size": f"{FULL_QUERIES:,} queries, {dcn_rows_full:,} rows", "one epoch (min)": round(dcn_epoch_full / 60, 1), "plan": "3 epochs + feature building", "total (min)": f"{dcn_full_3 / 60:.0f}", "fits in ~60 min": "yes" if dcn_full_3 / 60 <= 60 else "no"},
    {"model": "DCN-V2 + MMoE", "full-window training size": "same", "one epoch (min)": round(dcn_epoch_full / 60, 1), "plan": "10 epochs (as in #27) + feature building", "total (min)": f"{dcn_full_10 / 60:.0f}", "fits in ~60 min": "yes" if dcn_full_10 / 60 <= 60 else "no"},
])
with pl.Config(tbl_width_chars=220, fmt_str_lengths=60, tbl_cols=10):
    print(est)

In [ ]:
# memory: the DCN tensors grow with the rows
dcn_mem_full = dcn["peak_memory_gb"] * (dcn_rows_full / dcn["rows"])
print(f"DCN peak memory scales with rows: {dcn['peak_memory_gb']} GB at {dcn['rows']:,} rows -> about {dcn_mem_full:.0f} GB at {dcn_rows_full:,} rows (does not fit in {round(ram)} GB unless streamed in chunks)")
summary = {"hardware": hardware, "measured": [tt, dcn], "estimate_minutes": {
    "two_tower_1_to_2_epochs": [round(tt_full / 60), round(tt_full2 / 60)], "dcn_3_epochs": round(dcn_full_3 / 60), "dcn_10_epochs": round(dcn_full_10 / 60)},
    "dcn_full_memory_gb_estimate": round(dcn_mem_full)}
(ROOT / "data" / "results" / "cpu_training_time.json").write_text(json.dumps(summary, indent=2))
print("saved data/results/cpu_training_time.json")

## Conclusion

**Hardware:** Apple M3, 8 cores, 8 GB RAM, macOS on arm64. PyTorch used the CPU with 4 threads (the other 4 cores were not used; more threads were not tried).

| Model | Training size | Epochs | Train time | Per epoch | Peak memory | Quality reached |
|---|---|---|---|---|---|---|
| Two-Tower | 382,261 queries, 641,425 examples | 4 | 332 s | 83 s | 1.66 GB | validation loss 3.38, eval recall@100 0.291 |
| DCN-V2 + MMoE | 36,000 queries, 7.2 million rows | 3 | 92 s | 31 s | 1.45 GB | validation loss 0.031, AUC 0.83 / 0.87 / 0.88 |

(The Two-Tower sampled 500,000 queries and 382,261 remained after the relevance filter. Data preparation before training took 72 s for the Two-Tower and 4 s for DCN.)

**At the sizes we actually use, both fit easily:** a few minutes and under 2 GB each.

**Full-window estimate** (890,042 usable queries, linear scaling):
- **Two-Tower: fits.** About 3 minutes per epoch, so 3 to 6 minutes for the 1 to 2 epochs that are useful (the best epoch was epoch 1 in #24 and #25). Memory stays small because each batch only touches the items in it.
- **DCN-V2 + MMoE: time is borderline, memory does not fit.** The full window is 178 million candidate rows: 12.6 minutes per epoch, so **58 minutes for 3 epochs** including feature building, and **146 minutes for the 10 epochs** used in #27. The tensors would need about **36 GB**, more than this Mac's 8 GB, so a full-size run must stream the data in chunks instead of loading it whole.
- **What would be reduced** to stay under an hour and 8 GB: train on a sample of queries, as here (40,000 queries already give the reported results); or keep fewer candidates per query (for example 50 instead of 200, a 4-fold cut in rows); or stream from parquet in chunks; and use 3 to 5 epochs. A spot GPU or cloud CPU job (the fallback in the plan) is only needed if a full-size DCN run is really wanted.

**Caveat on the timings.** They are not stable on this machine: the same Two-Tower training took about 82 seconds in #24 and 255 to 332 seconds in later runs (#25 and here), presumably because other work was running at the same time or the chip throttled. Treat the times as a range up to about 4 times faster than shown, and the estimates as upper bounds on this hardware. Peak memory is stable.

## What I learned

To be written by the owner of the project.